In [1]:
import json
import numpy as np
from tqdm import tqdm
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
import numpy as np
from scipy.optimize import least_squares
from scipy.special import erf

In [2]:
index = {}

with open("../Step_2_Integration/observability_bright.json", "rb") as f:
    while True:
        pos = f.tell()
        line = f.readline()

        if not line:
            break

        s = line.strip()

        if s in (b"", b"{", b"}"):
            continue

        # key is before first colon
        key_bytes = s.split(b":", 1)[0]
        key = json.loads(key_bytes.decode())

        index[int(key)] = pos

print("nkeys:", len(index))

def load_observability_key(filename, index, key):
    key = int(key)

    with open(filename, "rb") as f:
        f.seek(index[key])
        line = f.readline().decode().strip()

    if line.endswith(","):
        line = line[:-1]

    rec = json.loads("{" + line + "}")
    return np.asarray(rec[str(key)])

nkeys: 51190


In [11]:
def generate_trail(shape, flux, x0, y0, length, theta, sigma, background=0.0):
    ny, nx = shape
    y, x = np.mgrid[0:ny, 0:nx]
    xp = (x - x0) * np.cos(theta) + (y - y0) * np.sin(theta)
    yp = -(x - x0) * np.sin(theta) + (y - y0) * np.cos(theta)
    prefactor = flux / (length * 2.0 * sigma * np.sqrt(2.0 * np.pi))
    cross_trail = np.exp(-(yp ** 2) / (2.0 * sigma ** 2))
    u1 = (xp + length / 2.0) / (sigma * np.sqrt(2.0))
    u2 = (xp - length / 2.0) / (sigma * np.sqrt(2.0))
    along_trail = erf(u1) - erf(u2)
    return background + prefactor * cross_trail * along_trail


def fit_trail(image, p0, bounds=None):
    ny, nx = image.shape

    def residuals(params):
        flux, x0, y0, length, theta, sigma, background = params
        if length <= 0 or sigma <= 0 or flux < 0:
            return np.full(image.size, 1e30)
        model = generate_trail(image.shape, flux, x0, y0, length, theta, sigma, background)
        return (model - image).ravel()

    if bounds is None:
        lower = [0.0, 0.0, 0.0, 1e-6, -np.pi, 1e-3, -np.inf]
        upper = [np.inf, nx, ny, max(nx, ny), np.pi, np.inf, np.inf]
        bounds = (lower, upper)

    result = least_squares(residuals, p0, bounds=bounds)
    best_fit = {"flux": result.x[0], "x0": result.x[1], "y0": result.x[2], "length": result.x[3], "theta": result.x[4], "sigma": result.x[5], "background": result.x[6]}
    model = generate_trail(image.shape, best_fit["flux"], best_fit["x0"], best_fit["y0"], best_fit["length"], best_fit["theta"], best_fit["sigma"], best_fit["background"])
    return result, best_fit, model

In [13]:
mpcnum = 1
obs = load_observability_key("../Step_2_Integration/observability_bright.json", index, mpcnum)
print(obs.shape)
print(obs[:10])

(6263, 7)
[['243.33128' '-12.933481' '2.2607365' '2.6862247' '2411433.917859954'
  'i00768' '7.246961421394394']
 ['230.25215' '-15.6733885' '2.193429' '2.7920167' '2411565.614578704'
  'i01460' '7.265207963672594']
 ['339.79263' '-19.649696' '2.4535859' '2.9757347' '2411898.912928241'
  'b06327' '7.646977382690467']
 ['340.133' '-19.848923' '2.4063885' '2.9764988' '2411902.8112824075'
  'b06363' '7.605357276225835']
 ['340.40335' '-20.092997' '2.3588226' '2.9772534' '2411906.865460648'
  'b06387' '7.562555400081958']
 ['340.40387' '-20.09358' '2.3587186' '2.977255' '2411906.874462963'
  'b06388' '7.562460825168908']
 ['340.1122' '-22.242989' '2.1230605' '2.9808629' '2411930.759415509'
  'b06561' '7.336521860361453']
 ['50.76535' '12.331429' '1.8360833' '2.759565' '2412438.656556713'
  'i07633' '6.853665043017613']
 ['50.00687' '12.441715' '1.8578041' '2.756414' '2412442.6348993056'
  'i07647' '6.876721820241446']
 ['48.881805' '12.694613' '1.9050819' '2.7508905' '2412449.6057453705'
 

In [ ]:
fit_results = []

for i in range(1): #loop over the images, plot and fit them!
    ra, dec, rearth, rsun, jd, plate_id, vmag = obs[i]
    fits_path = f"./{mpcnum}/{mpcnum}_{plate_id}.fits"
    hdul = fits.open(fits_path)
    hdu = next(h for h in hdul if h.data is not None)

    image = hdu.data
    wcs = WCS(hdu.header)
    
    ny, nx = image.shape

    p0 = [
        data[int(nx/2),int(ny/2)],            # flux
        nx / 2,            # x0
        ny / 2,            # y0
        15.0,              # length
        np.deg2rad(30.0),  # theta
        1.5,               # sigma
        np.median(image),  # background
    ]

    result, best_fit, model = fit_trail(image, p0)
    resid = image - model

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    vmin = min(np.min(image), np.min(model))
    vmax = max(np.max(image), np.max(model))
    rmax = np.max(np.abs(resid))

    axes[0].imshow(image, origin="lower", vmin=vmin, vmax=vmax)
    axes[0].set_title("Data")

    axes[1].imshow(model, origin="lower", vmin=vmin, vmax=vmax)
    axes[1].set_title("Fit")

    axes[2].imshow(resid, origin="lower", vmin=-rmax, vmax=rmax)
    axes[2].set_title("Residual")

    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()
    
    fit_results.append(result,best_fit,model)